# 00 · Introduction: LangGraph

Welcome. In this tutorial you will build two working agents with LangGraph: **ShipFast**, a support agent
that looks up orders and issues refunds (and asks a person before any refund over $100), and a **research
agent** that searches, drafts an answer and critiques its own draft. You will get there one idea at a time,
from a single model call in notebook 01 to the finished agents in notebook 08.

First, though, it helps to know what LangGraph is and how it thinks about agents, because that explains
most of the design choices you will meet along the way. At the end you will check that your setup works.

## What it is

**LangGraph** is an open-source Python (and JavaScript) library from LangChain. In its own words, it is
"a low-level orchestration framework for building, managing, and deploying long-running, stateful agents",
providing "the infrastructure for durable execution, streaming, human-in-the-loop, persistence" and more.
LangChain's own agents are built on top of it: `create_agent`, which notebook 03 uses, is a ready-made
LangGraph graph. Its design is inspired by Google's Pregel and Apache Beam.

The README is also clear about when to reach for it: "when you have advanced needs that require a
combination of deterministic and agentic workflows, heavy customization, and carefully controlled
latency". For a plain agent loop, LangChain's agents (built on LangGraph) are the simpler entry point.

> **Versions.** The quotes are from LangGraph's README. Everything in this tutorial uses LangGraph 1.2.19
> with langchain 1.4.8; newer versions may differ in the details.

## The core idea

![LangGraph: the core idea](images/langgraph_core_idea.svg)

LangGraph's idea is that an agent system is a **graph you draw**:

- **State**: one typed record (here, the conversation's messages and the triage result) that every step
  reads and updates.
- **Nodes**: the steps. A node is a plain function or a model call, and it can even be a whole agent.
- **Edges**: what runs next. A **conditional edge** calls a function that picks the next node, so routing
  can be decided by code (out of scope? decline) or by the model.
- **Checkpointer**: saves the state after every step, per conversation thread. That one mechanism gives you
  memory across turns, pausing for a person and resuming later, and replaying a run to debug it.

You decide how much is fixed and how much the model decides. That is the trade: you draw more yourself, and
in return you know exactly what can happen.

## How the course framed it

In Module 4 of the Agentic AI Crash Course, the "LangGraph vs CrewAI" slide sums LangGraph up like this: its mental model is "a graph of
steps over shared state"; control is "fine-grained: you draw every edge"; its strengths are "durable state,
checkpoints, human interrupts, time travel"; and the thing to watch for is "more concepts and code up front".
A good first project, the slide suggests, is "a long-running workflow with approvals". ShipFast's refund
approval is exactly that kind of workflow, so keep an eye on it in notebook 06.

## What we found building with it

We built the same two agents in LangGraph, CrewAI and Strands, with the same tools, data and local model,
and scored them the same way. Here is how LangGraph did:

- It had the best results here: ShipFast 7 of 8 and research 10 of 10.
- Running fully locally needed nothing switched off: no telemetry, no cloud defaults.
- The approval gate pauses **inside the tool call**, with `interrupt()`, and resumes with a `Command`.
- One practical catch: middleware written as a plain function is synchronous only, so the async capstone
  runs the graph in a worker thread.

Treat these as a first look rather than a verdict: each case ran once, with a small model on one laptop.
The full side-by-side is in `COMPARISON.md` at the repo root.

## The concepts, and where you meet them

| Concept | Notebook |
|---|---|
| Chat model, structured output (`ChatOllama`, `with_structured_output`) | 01 |
| Tools (`tool`, `bind_tools`, tool calls) | 02 |
| The agent loop (`create_agent`, `recursion_limit`) | 03 |
| State, checkpointer, thread id | 04 |
| `StateGraph`: nodes, edges, conditional edges | 05 |
| Middleware, `interrupt`, `Command(resume=...)` | 06 |
| Callbacks for tracing and metrics | 07 |
| Putting it together: ShipFast and the research agent | 08 |

## How this tutorial works

![The path through every tutorial](images/roadmap.svg)

- **Same path in every framework.** The LangGraph, CrewAI and Strands tutorials have the same notebooks in
  the same order, so you can read one notebook in two frameworks side by side. Each notebook ends with
  "How the others do this".
- **Two running examples.** **ShipFast**, a customer-support agent for a made-up shop (orders, refund
  policy, a human approval for refunds over $100, memory across turns), and a **research agent** that answers
  questions about a made-up company from fixed articles, with a critique loop.
- **One shared contract.** The data, tools, instructions, scoring and evaluation runner live once, in
  `contracts/`; this tutorial wraps them in LangGraph. Its reusable code lives in `lg_agents/`, and both the
  notebooks and `run_eval.py` import it, so the code you read is the code that was tested.
- **One local model.** `qwen3-agents` (Qwen 3 8B) on Ollama, with fixed settings, so differences come from
  the framework. No API key is needed.
- **Real outputs, mistakes included.** Every notebook has been run, and the notes describe what actually
  happened, including where the small model got things wrong. Those mistakes are part of the lesson. When
  you run a notebook yourself, expect the details to differ a little.

## Setup check

You need Ollama running (`ollama serve`) and the shared model created once
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`; see the README). Pick this tutorial's kernel
(`.venv/bin/python` in this tutorial's folder) and run the cell below.

In [1]:
from importlib.metadata import version

from agent_contracts.harness import load_settings, verify_model

verify_model()
print("model:", load_settings().model)
print("langgraph", version("langgraph"), "| langchain", version("langchain"), "| langchain-ollama", version("langchain-ollama"))

model: qwen3-agents
langgraph 1.2.19 | langchain 1.4.8 | langchain-ollama 1.1.0


If you see the model name and the versions, you are all set. Next up: **01 · The model**, where you make
your first call.